# SimpleCNN Colab Notebook

This notebook trains the SimpleCNN baseline from scratch on FER2013.

Transfer learning is **not required** for the core project structure, so it is kept as an optional comparison only.

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix, classification_report
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

ROOT = Path('/content/EE4016_MoodMirror')
if IN_COLAB and not ROOT.exists():
    print('Optional: clone your repo into /content/EE4016_MoodMirror before running training cells.')

PROJECT_ROOT = ROOT if ROOT.exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Colab environment:', IN_COLAB)

## Load Dataset and Preview Samples

The notebook expects FER2013 in the existing folder layout at `data/fer2013`.
If you are in Colab, mount Drive or copy the dataset into that path before training.

In [ ]:
from src.data_loader import get_dataloaders, EMOTIONS

DATA_PATH = PROJECT_ROOT / 'data' / 'fer2013'
BATCH_SIZE = 64

if not DATA_PATH.exists():
    raise FileNotFoundError(f'FER2013 dataset not found at {DATA_PATH}')

train_loader, val_loader, test_loader = get_dataloaders(
    str(DATA_PATH),
    batch_size=BATCH_SIZE,
    pin_memory=torch.cuda.is_available(),
)

print('Classes:', EMOTIONS)
print('Train batches:', len(train_loader))
print('Validation batches:', len(val_loader))
print('Test batches:', len(test_loader))

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, label in zip(axes.flat, images[:8], labels[:8]):
    axis.imshow(image.squeeze(0), cmap='gray')
    axis.set_title(EMOTIONS[int(label)])
    axis.axis('off')
plt.tight_layout()

## Keep Jasmine's Original Code for Reference

Do not edit the original branch source. For reference, the original model implementation lives in `jasmine_code/src/models.py`.
This notebook keeps that branch untouched and trains the baseline separately.

## Preprocess and Prepare Data

The existing project data loader already handles resizing, normalization, augmentation, and train/validation/test splits.
This notebook uses that shared pipeline rather than redefining the preprocessing steps.

In [ ]:
from src.models import SimpleCNN, count_parameters

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)

model = SimpleCNN(num_classes=len(EMOTIONS)).to(DEVICE)
print('Trainable parameters:', f'{count_parameters(model):,}')

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-4)


## Train the Baseline Simple CNN

This is the main deliverable path for SimpleCNN. It trains from scratch, uses GPU acceleration when available, and saves the best checkpoint.

In [ ]:
SAVE_DIR = PROJECT_ROOT / 'saved_models'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = SAVE_DIR / 'SimpleCNN_best_colab.pth'
EPOCHS = 50
USE_AMP = DEVICE.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

best_val_acc = 0.0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_start = time.time()
    for inputs, targets in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}', leave=False):
        inputs = inputs.to(DEVICE, non_blocking=USE_AMP)
        targets = targets.to(DEVICE, non_blocking=USE_AMP)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        train_correct += (outputs.argmax(dim=1) == targets).sum().item()
        train_total += targets.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(DEVICE, non_blocking=USE_AMP)
            targets = targets.to(DEVICE, non_blocking=USE_AMP)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            val_correct += (outputs.argmax(dim=1) == targets).sum().item()
            val_total += targets.size(0)

    scheduler.step()

    train_acc = 100.0 * train_correct / train_total
    val_acc = 100.0 * val_correct / val_total
    avg_train_loss = train_loss / train_total
    avg_val_loss = val_loss / val_total

    print(
        f'Epoch {epoch + 1:02d}/{EPOCHS} | '
        f'Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
        f'Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict()
        torch.save(best_state, BEST_MODEL_PATH)
        print(f'Best checkpoint saved to {BEST_MODEL_PATH}')

print('Best validation accuracy:', f'{best_val_acc:.2f}%')

## Evaluate Baseline Performance

This section reloads the best checkpoint, evaluates on the test split, and prints the main report metrics.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

best_model = SimpleCNN(num_classes=len(EMOTIONS)).to(DEVICE)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
best_model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(DEVICE, non_blocking=USE_AMP)
        targets = targets.to(DEVICE, non_blocking=USE_AMP)
        outputs = best_model(inputs)
        predictions = outputs.argmax(dim=1)
        y_true.extend(targets.cpu().numpy().tolist())
        y_pred.extend(predictions.cpu().numpy().tolist())

accuracy = (np.array(y_true) == np.array(y_pred)).mean() * 100.0
print(f'Test accuracy: {accuracy:.2f}%')
print(classification_report(y_true, y_pred, target_names=EMOTIONS, digits=4))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=EMOTIONS)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.xticks(rotation=45)
plt.tight_layout()

## Optional: Transfer Learning Comparison

This is a separate comparison only. It is not part of the core project structure for SimpleCNN.
Leave it disabled unless you specifically want to test whether a pretrained backbone improves the result.

In [ ]:
USE_TRANSFER_LEARNING = False

if USE_TRANSFER_LEARNING:
    from torchvision import models, transforms
    from torch.utils.data import DataLoader

    transfer_train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Lambda(lambda image: image.convert('RGB')),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    transfer_test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Lambda(lambda image: image.convert('RGB')),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    print('Transfer learning is enabled, but it is only a comparison path.')
    print('Keep this off unless you want to benchmark against a pretrained model.')
else:
    print('Transfer learning comparison is disabled.')

## Compare Baseline vs. Transfer Learning

If you enable the optional section, compare its validation and test metrics here.
Otherwise, the notebook will clearly show that the baseline SimpleCNN is the chosen path.

## Save Model and Run Inference

The best SimpleCNN checkpoint is already saved during training. This final section shows how to reload it and run inference on a new image.

In [ ]:
def predict_image(image_path, model, class_names):
    from PIL import Image
    from torchvision import transforms

    image = Image.open(image_path).convert('L')
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])
    tensor = transform(image).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probabilities = torch.softmax(logits, dim=1).squeeze(0)
        predicted_index = int(probabilities.argmax().item())

    return class_names[predicted_index], float(probabilities[predicted_index].item())

sample_path = None
print('Checkpoint saved at:', BEST_MODEL_PATH)
print('Use predict_image("path/to/image.jpg", best_model, EMOTIONS) to run inference.')

## Final Notes

SimpleCNN is the baseline deliverable and is trained from scratch.
Transfer learning is optional, not required by the current project structure, and should only be used as a comparison if the baseline does not reach the target.